# Supervised Anomaly Classification

## Objective
Train and evaluate supervised classifiers on both datasets:
1. **Random Forest** — with class weight balancing
2. **XGBoost** — with scale_pos_weight for imbalance handling
3. **Gradient Boosting** — alternative boosting baseline
4. **Voting Ensemble** — soft-voting combination of all three

Evaluation on both UCI Grid Stability and Synthetic Power Plant datasets.

In [ ]:
import sys, os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.abspath('..'))

from src.models import train_supervised_models
from src.evaluation import (
    evaluate_supervised, plot_confusion_matrices,
    plot_roc_pr_curves, plot_feature_importance,
    print_classification_report,
)
from src.data_loader import split_data

DATA_DIR = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results', 'figures')
print('Libraries loaded')

## 5.1 Power Plant Dataset — Supervised

In [ ]:
plant_df = pd.read_csv(os.path.join(DATA_DIR, 'plant_engineered.csv'))

with open(os.path.join(DATA_DIR, 'engineered_feature_names.json'), 'r') as f:
    feat_names = json.load(f)

feature_cols = [c for c in feat_names['all'] if c in plant_df.columns]
plant_clean = plant_df.dropna(subset=feature_cols).reset_index(drop=True)

X_train, X_test, y_train, y_test = split_data(plant_clean, feature_cols)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols)

# Compute class imbalance ratio for XGBoost
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f'Scale pos weight: {scale_pos_weight:.2f}')

In [ ]:
print('\n=== Training Supervised Models (Power Plant) ===')
plant_sup_results = train_supervised_models(X_train_scaled, y_train, scale_pos_weight)

In [ ]:
plant_sup_eval = evaluate_supervised(plant_sup_results, X_test_scaled, y_test)
print('\nPower Plant — Supervised Results:')
print(plant_sup_eval[['precision', 'recall', 'f1_score', 'mcc', 'roc_auc', 'avg_precision', 'train_time']].round(4).to_string())

In [ ]:
plot_confusion_matrices(plant_sup_results, X_test_scaled, y_test,
                        save_path=os.path.join(RESULTS_DIR, 'plant_supervised_confusion.png'))

In [ ]:
plot_roc_pr_curves(plant_sup_results, X_test_scaled, y_test,
                   save_path=os.path.join(RESULTS_DIR, 'plant_supervised_roc_pr.png'))

In [ ]:
# Feature importance from best model
best_name = plant_sup_eval['f1_score'].idxmax()
best_model = plant_sup_results[best_name]['model']
plot_feature_importance(best_model, feature_cols, top_n=25,
                        save_path=os.path.join(RESULTS_DIR, 'plant_feature_importance.png'))

## 5.2 UCI Grid Stability — Supervised

In [ ]:
uci_df = pd.read_csv(os.path.join(DATA_DIR, 'uci_grid_processed.csv'))
uci_features = ['tau1','tau2','tau3','tau4','p1','p2','p3','p4','g1','g2','g3','g4']

X_train_u, X_test_u, y_train_u, y_test_u = split_data(uci_df, uci_features)

scaler_u = StandardScaler()
X_train_u_scaled = pd.DataFrame(scaler_u.fit_transform(X_train_u), columns=uci_features)
X_test_u_scaled = pd.DataFrame(scaler_u.transform(X_test_u), columns=uci_features)

scale_pw_u = (y_train_u == 0).sum() / max((y_train_u == 1).sum(), 1)
print(f'UCI scale pos weight: {scale_pw_u:.2f}')

In [ ]:
print('\n=== Training Supervised Models (UCI Grid) ===')
uci_sup_results = train_supervised_models(X_train_u_scaled, y_train_u, scale_pw_u)

In [ ]:
uci_sup_eval = evaluate_supervised(uci_sup_results, X_test_u_scaled, y_test_u)
print('\nUCI Grid — Supervised Results:')
print(uci_sup_eval[['precision', 'recall', 'f1_score', 'mcc', 'roc_auc', 'avg_precision', 'train_time']].round(4).to_string())

In [ ]:
plot_confusion_matrices(uci_sup_results, X_test_u_scaled, y_test_u,
                        save_path=os.path.join(RESULTS_DIR, 'uci_supervised_confusion.png'))

## 5.3 Save Results

In [ ]:
plant_sup_eval.to_csv(os.path.join(DATA_DIR, 'plant_supervised_results.csv'))
uci_sup_eval.to_csv(os.path.join(DATA_DIR, 'uci_supervised_results.csv'))
print('Saved supervised results for both datasets')

## Summary

Supervised models significantly outperform unsupervised baselines when labeled data is available. The Voting Ensemble and XGBoost with class weighting show the strongest performance.

